<a href="https://colab.research.google.com/github/davidrpugh/introduction-to-deep-learning/blob/master/notebooks/01b-mlp-for-regression-with-pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multi-Layer Perceptrons (MLPs) for Regression with PyTorch

In [ ]:
import numpy as np
import torch
from torch import nn, optim

from sklearn import compose, datasets, metrics, model_selection
from sklearn import pipeline, preprocessing

## Loading the data

In [ ]:
housing_dataset = datasets.fetch_california_housing(
    as_frame=True
)

In [ ]:
print(housing_dataset["DESCR"])

In [ ]:
housing_features_df = housing_dataset["data"]
median_house_value = housing_dataset["target"]

In [ ]:
housing_features_df.info()

In [ ]:
_ = median_house_value.hist()

## Preparing the data

### Train/Val Split

In [ ]:
RANDOM_STATE = np.random.RandomState(42)


train_features_df, val_features_df, train_target, val_target = (
    model_selection.train_test_split(
        housing_features_df,
        median_house_value,
        test_size=0.20,
        shuffle=True,
        random_state=RANDOM_STATE
    )
)


In [ ]:
train_features_df.info()

In [ ]:
val_features_df.info()

### Features and target preparation

In [ ]:
def array_to_tensor(arr, dtype=torch.float32):
  return torch.tensor(arr, dtype=dtype)


def dataframe_to_tensor(df, dtype=torch.float32):
    arr = df.to_numpy()
    return array_to_tensor(arr, dtype)


def series_to_tensor(s, dtype=torch.float32):
    df = s.to_frame()
    return dataframe_to_tensor(df, dtype)


prepare_housing_features = pipeline.make_pipeline(
    preprocessing.QuantileTransformer(),
    preprocessing.FunctionTransformer(
        func=array_to_tensor
    )
)

prepare_housing_target = pipeline.make_pipeline(
    preprocessing.FunctionTransformer(
        func=series_to_tensor
    )
)



In [ ]:
X_train = prepare_housing_features.fit_transform(train_features_df)
X_val = prepare_housing_features.transform(val_features_df)


In [ ]:
print(X_train.shape)
print(X_val.shape)

In [ ]:
y_train = prepare_housing_target.fit_transform(train_target)
y_val = prepare_housing_target.transform(val_target)


In [ ]:
print(y_train.shape)
print(y_val.shape)

## Implementing a MLP for Regression using nn.Sequential

[`nn.Sequential`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html) in PyTorch is a container module that allows for the sequential execution of a series of neural network layers or modules. It simplifies the process of building neural networks with a linear, feed-forward structure by eliminating the need to explicitly define the forward method for each layer.


### Key Characteristics and Use-Cases

* **Ordered Container:** `nn.Sequential` takes a list of `nn.Module` instances (layers) and arranges them in the order they are provided.
* **Automatic Forward Pass:** When an input tensor is passed to an `nn.Sequential` object, it automatically propagates through each contained module in the defined order, with the output of one module serving as the input to the next.
* **Simplified Model Definition:** It offers a concise way to define models, especially for straightforward architectures without complex branching or custom logic within the forward pass.
* **Treat as a Single Module:** The entire `nn.Sequential` container can be treated as a single `nn.Module`, allowing for easy integration into larger models or for applying operations like moving to a device (`.to(device)`) or setting training/evaluation mode (`.train()`, `.eval()`).

In [ ]:
_ = torch.manual_seed(42)

_, n_features = X_train.shape
housing_model = nn.Sequential(
    nn.Linear(
        in_features=n_features,
        out_features=50,
        bias=True,
    ),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, 1)
)


### Layer-by-Layer Explanation

1. **Input layer:**
  * Input: Number of inputs (`n_features`)
  * Output: Number of neurons in the first hidden layer (50 neurons) → a tunable hyperparameter
2. **Activation:**
  * [`nn.ReLU`](https://docs.pytorch.org/docs/stable/generated/torch.nn.ReLU.html)
  * Applies ReLU elementwise (no parameters, same input/output shape)
3. **Second hidden layer:**
  * Input: 50 neurons (must match previous output!)
  * Output: Number of neurons in the second hidden layer (40 neurons) → a tunable hyperparameter
4. **Activation:**
  * `nn.ReLU`
5. **Output layer:**
  * Input: 40 neurons (must match previous output!)
  * Output: 1 neuron (must match the regression target dimensionality!)

### Loss functions and optimizers

In [ ]:
mse_loss = nn.MSELoss()

sgd = optim.SGD(
    housing_model.parameters(),
    lr=1e-2
)

### Define a training loop


In [ ]:
def train(
    model_fn,
    criterion,
    optimizer,
    X_train,
    y_train,
    X_val,
    y_val,
    n_epochs,
    log_epochs=100,
    ):

    for epoch in range(n_epochs):
        # forward pass
        y_pred = model_fn(X_train)
        train_loss = criterion(y_pred, y_train)

        # backward pass
        train_loss.backward()

        # gradient descent step
        optimizer.step()
        optimizer.zero_grad()

        # evaluate using the validation data
        with torch.no_grad():
            y_pred = model_fn(X_val)
            val_loss = criterion(y_pred, y_val)

        if (epoch + 1) % log_epochs == 0:
            print(f"Epoch {epoch + 1}/{n_epochs}, Training Loss: {train_loss.item(): .4f}, Val Loss: {val_loss.item(): .4f}")



In [ ]:
train(
    housing_model,
    mse_loss,
    sgd,
    X_train,
    y_train,
    X_val,
    y_val,
    n_epochs=1000
)